In [26]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from trodes import read_exported as tre
sys.path.append(r'c:\Users\zhaoz\Desktop\Research\SocialMemory\diff_fam_social_memory_ephys')

In [27]:
def pickle_this(thing_to_pickle, file_name):
    """
    Pickles things
    Args (2):   
        thing_to_pickle: anything you want to pickle
        file_name: str, filename that ends with .pkl 
    Returns:
        none
    """
    with open(file_name,'wb') as file:
        pickle.dump(thing_to_pickle, file)

def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):   
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, 'rb') as file:
        return(pickle.load(file))

In [3]:
path = r"C:\Users\zhaoz\Desktop\Research\Cooperation\ecu_behaviors"

box_to_ecu_dict = {
            1: {'dio_ECU_Din1': 'selfish light',
                'dio_ECU_Din2': 'coop light',
                'dio_ECU_Din6': 'selfish nose poke',
                'dio_ECU_Din10': 'coop nose poke',
                'dio_ECU_Din8': 'subject port entry',
                'dio_ECU_Din16': 'recipient port entry'},   
            2: {'dio_ECU_Din3': 'selfish light',
                'dio_ECU_Din4': 'coop light',
                'dio_ECU_Din22': 'selfish nose poke',
                'dio_ECU_Din26': 'coop nose poke',
                'dio_ECU_Din24': 'subject port entry',
                'dio_ECU_Din32': 'recipient port entry'}}

# Read the box distributions for different recordings
df = pd.read_csv(path + "\Coop_ephys_Box_organization(Sheet1).csv")
df["Individual name"] = df["Individual name"].astype(str) + "_merged"
print(df.to_string)

<bound method DataFrame.to_string of                              Individual name         Trainning  Box  \
0   20250517_115803_alternates_D1_1-2_merged  Stage 3 last day  1.0   
1   20250517_115803_alternates_D1_1-3_merged  Stage 3 last day  1.0   
2   20250517_115803_alternates_D1_2-4_merged  Stage 3 last day  2.0   
3   20250517_115803_alternates_D1_2-1_merged  Stage 3 last day  2.0   
4   20250507_103358_alternates_D1_6-1_merged  Stage 3 last day  1.0   
..                                       ...               ...  ...   
82      20250517_134054_Stage4_D8_4-2_merged      Stage4 day 8  2.0   
83      20250517_134054_Stage4_D8_4-3_merged      Stage4 day 8  1.0   
84      20250517_134054_Stage4_D8_4-1_merged      Stage4 day 8  1.0   
85      20250517_121746_Stage4_D8_6-1_merged      Stage4 day 8  1.0   
86      20250517_121746_Stage4_D8_6-3_merged      Stage4 day 8  2.0   

   Mice condition  
0         Subject  
1       Recipient  
2         Subject  
3       Recipient  
4         

In [14]:
def trodes_behaviors_to_timestamps(name, data, clockrate, first_timestamp):
	'''
	Convert the data in .dat file format to normal behavior numpy array
	'''
	behavior_array = []
	event = []
	status = 1

	for (time, value) in data[1:]:
		if value[0] != status and not name == "20250516_111920_Stage4_D7_2-1_merged" and not name == "20250516_111920_Stage4_D7_2-4_merged":
			raise ValueError('Invalid timestamp - ' + name + ': ' + str(time[0]))
		
		event.append((time[0] - int(first_timestamp))/int(clockrate)) #align behavior data to 0
		if (status == 0):
			behavior_array.append(event)
			event = []
		status = 1 - status
	
	return np.array(behavior_array, dtype=float).tolist()

In [15]:
def print_n_o_events(behaviors_dict):
	for key, value in behaviors_dict.items():
		print('#', key, "-", len(value))

In [34]:
def load_trodes_recording(absolute_path):
	'''
	Read the relative path to the folder that contains merged.DIO folder
	'''
	name = Path(absolute_path).stem #20250508_100203_Stage4_D1_1-2_merged
	# absolute_path = os.path.join(path, relative_path)
	# name = relative_path.split('.')[0]	#20250508_100203_Stage4_D1_1-2_merged
	box = int(df.loc[df["Individual name"] == name, "Box"].iloc[0])	#1 or 2
	behaviors_dict = {}

	# Walk through the .dat files
	for root, dirs, files in os.walk(absolute_path):
		for file in files:
			if file.endswith('.dat'):
				din = file.split('.')[1]
				if din in box_to_ecu_dict[box]:	#if the din channel is meaningful
					data = tre.read_trodes_extracted_data_file(os.path.join(absolute_path, file))
					event = box_to_ecu_dict[box][din]
					behavior_data = data['data']
					if event == 'coop light':
						print(data['clockrate'])
						print(behavior_data)
					# print(event, len(behavior_data))
					# print(data['first_timestamp'])
					# print(behavior_data)
					behavior_array = trodes_behaviors_to_timestamps(name, behavior_data, data['clockrate'], data['first_timestamp'])
					behaviors_dict[event] = behavior_array
					behaviors_dict = dict(sorted(behaviors_dict.items()))

	return (name, behaviors_dict)

In [35]:
# Change it to the corresopnding recording
rec_name, behaviors = load_trodes_recording(r"C:\Users\zhaoz\Desktop\Research\Cooperation\ecu_behaviors\Stage4_D8\20250517_121746_Stage4_D8_6-1_merged.DIO")

print(rec_name)
# print(behaviors)
print_n_o_events(behaviors)

20000
[([ 1469021], [0]) ([ 4877501], [1]) ([ 4957509], [0]) ([ 5300306], [1])
 ([ 5380307], [0]) ([ 5613111], [1]) ([ 5693116], [0]) ([ 5734912], [1])
 ([ 5814913], [0]) ([ 5850513], [1]) ([ 5930514], [0]) ([ 5976915], [1])
 ([ 6056921], [0]) ([ 6093916], [1]) ([ 6173923], [0]) ([ 6216918], [1])
 ([ 6296919], [0]) ([ 6360122], [1]) ([ 6440121], [0]) ([ 6494321], [1])
 ([ 6574322], [0]) ([ 6629723], [1]) ([ 6709731], [0]) ([ 6804133], [1])
 ([ 6884126], [0]) ([ 6933327], [1]) ([ 7013328], [0]) ([ 7064329], [1])
 ([ 7144330], [0]) ([ 7195930], [1]) ([ 7275936], [0]) ([ 7348532], [1])
 ([ 7428535], [0]) ([ 7516934], [1]) ([ 7596933], [0]) ([ 7624735], [1])
 ([ 7704737], [0]) ([ 7749937], [1]) ([ 7829943], [0]) ([ 8208143], [1])
 ([ 8288142], [0]) ([15936050], [1]) ([16016045], [0]) ([16060051], [1])
 ([16140045], [0]) ([16251449], [1]) ([16331447], [0]) ([16373848], [1])
 ([16453848], [0]) ([16495049], [1]) ([16575053], [0]) ([22554527], [1])
 ([22634528], [0]) ([24965563], [1]) ([250455

# Number of Events

In [28]:
Stage4 = []
for root, dirs, files in os.walk(path):
	for file in files:
		if file.endswith('.pkl'):
			day_rec = unpickle_this(os.path.join(path, file))
			Stage4.append(day_rec)
print(len(Stage4))

8


In [31]:
for i in range (6, 8):
	print(f"Day {i+1}")
	print("------------------------------------")
	for rec, rec_behaviors in Stage4[i].items():
		print(rec)
		print_n_o_events(rec_behaviors)
		print()

Day 7
------------------------------------
20250516_095627_Stage4_D7_1-2_merged
# selfish light - 228
# coop nose poke - 241
# recipient port entry - 301
# coop light - 196
# selfish nose poke - 70
# subject port entry - 394

20250516_095627_Stage4_D7_1-3_merged
# selfish light - 228
# coop nose poke - 241
# recipient port entry - 301
# coop light - 196
# selfish nose poke - 70
# subject port entry - 394

20250516_111920_Stage4_D7_2-1_merged
# selfish nose poke - 62
# subject port entry - 576
# coop nose poke - 171
# selfish light - 139
# recipient port entry - 376
# coop light - 93

20250516_111920_Stage4_D7_2-4_merged
# selfish nose poke - 62
# subject port entry - 576
# coop nose poke - 171
# selfish light - 139
# recipient port entry - 376
# coop light - 93

20250516_135729_Stage4_D7_4-1_merged
# selfish light - 183
# coop nose poke - 209
# recipient port entry - 313
# coop light - 163
# selfish nose poke - 29
# subject port entry - 1589

20250516_135729_Stage4_D7_4-2_merged
# self